# Commanded LUT and Trim during bending-mode CWFS tests (v1)

**Author:** Aaron Roodman  
**Created:** 2026-09-17 · **Last modified:** 2026-09-17  
**Status:** in progress  
**Keywords:** AOS, LUT, Trim, CWFS, bending modes, BLOCK-T377, BLOCK-T378, BLOCK-T379, BLOCK-T380, DuckDB

## Description

The commanded hexapod Look-Up Table (LUT) and the accumulated closed-loop Trim offset, per
visit, for the Corner Wavefront Sensor (CWFS) exposures taken under test BLOCKs T377, T378,
T379 and T380. These BLOCKs command individual mirror bending modes and take CWFS pairs, so
the interesting question is what the Active Optics System (AOS) had loaded and had
accumulated while each bending mode was being exercised.

Two commanded quantities, both read from the value-added telemetry DuckDB:

1. **Hexapod LUT** — 10 axes, `MTHexapod.logevent_compensationOffset`. This is the
   elevation / rotator / filter compensation the hexapods applied from their own LUT model,
   and it is what the Trim is measured *against*.
2. **Trim** — all 50 optical degrees of freedom (DOF), `MTAOS.logevent_degreeOfFreedom`.
   The accumulated closed-loop offset.

**Output:** figures inline; optionally a PDF under `output/bounce/`.

**References**
- [`docs/studies/bounce.md`](../../docs/studies/bounce.md) — the study this sits in
- [`docs/telemetry.md`](../../docs/telemetry.md) — every telemetry quantity, its EFD or ConsDB name, coverage and units
- `common/efd_db.py` — the DuckDB schema and read helpers
- `code/aos_trim.py:fetch_hexapod_lut_for_visits` — how the 10 LUT axes are ordered

## Change Log

| date | change |
|---|---|
| 2026-09-17 | created |

## Table of Contents

1. [Parameters](#params)
2. [Setup](#setup)
3. [Helper functions](#functions)
4. [Load the telemetry and select the programs](#data)
5. [Sample inventory](#inventory)
6. [Hexapod LUT, all 10 axes](#lut)
7. [Trim, all 50 DOF](#trim)
8. [A single DOF in detail](#single)
9. [Optional PDF](#pdf)

### Two things to know before reading the plots

**The LUT has 10 axes, the Trim has 50 DOF, and they are not the same index space** — but
the first ten do line up. Both order the hexapods M2 first, then camera:

| index | LUT axis (`lut_dofN`) | Trim DOF (`dofN`) |
|---|---|---|
| 0–4 | M2 hexapod z, x, y, u, v | `M2_dz`, `M2_dx`, `M2_dy`, `M2_rx`, `M2_ry` |
| 5–9 | camera hexapod z, x, y, u, v | `Cam_dz`, `Cam_dx`, `Cam_dy`, `Cam_rx`, `Cam_ry` |
| 10–29 | — | `B1_1`…`B1_20`, M1M3 bending |
| 30–49 | — | `B2_1`…`B2_20`, M2 bending |

So `lut_dof5` is the camera-hexapod dz LUT — the filter-dependent focus term — and it pairs
with Trim `dof5` = `Cam_dz`.

**The angular units differ between the two.** The LUT `u`/`v` axes are in **deg**, as the
hexapod reports them, while the Trim rotations `rx`/`ry` are in **arcsec**, the Optical
Feedback Control (OFC) convention. The translation axes are µm in both. Every panel title
carries its own unit; do not compare an angular LUT axis to an angular Trim DOF without
converting.

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================

# Test BLOCKs to select. Matched as a substring against ConsDB science_program, so
# 'T378' also picks up the suffixed variants ('BLOCK-T378_M1M3B12', 'BLOCK-T378_M1M3B20').
PROGRAMS = ('T377', 'T378', 'T379', 'T380')

# Image types to keep. These BLOCKs interleave 'acq' acquisition frames with the 'cwfs'
# pairs; set to None to keep every image type.
IMG_TYPES = ('cwfs',)

# day_obs range as (first, last), either bound None for open-ended. None covers the whole
# database, which is what the programs are allowed to select from.
DAY_OBS_RANGE = None

# Cache the ConsDB program/img_type join, which is the slow step (one query per night).
CACHE_META = True
CACHE_PATH = 'bounce/bending_mode_test_meta.parquet'   # relative to OUTPUT_ROOT

OUTPUT_ROOT = 'output'      # relative to the topic directory, resolved in Setup
SAVE_PDF = False            # True -> also write bounce/bending_mode_test_lut_trim.pdf

# The single DOF examined in detail in section 8. 5 = Cam_dz, the focus axis.
SINGLE_DOF = 5

# Plot style
GRID_FIGSIZE = (15.0, 7.5)   # one grid page
MS = 5                       # marker size

<a id='setup'></a>
## 2. Setup

In [ ]:
import sys
from pathlib import Path

# Walk up to the topic directory (the one holding code/), so this works at any depth.
_TOPIC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'code').is_dir())
sys.path.insert(0, str(_TOPIC.parent))        # repo root -> common/
sys.path.insert(0, str(_TOPIC / 'code'))      # flat cross-study modules

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from common import efd_db
from common.utils import nmad

# DOF labels, units and index groups come from the external package, so this notebook
# cannot drift from the convention the pipeline uses.
from lsst.ts.intrinsic.wavefront.ofc_svd import DOF_GROUPS, DOF_UNITS_50, LABELS_50DOF

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

OUT_DIR = Path(OUTPUT_ROOT)
if not OUT_DIR.is_absolute():
    OUT_DIR = _TOPIC / OUT_DIR
print(f'topic dir : {_TOPIC}')
print(f'output    : {OUT_DIR}')
print(f'database  : {efd_db.default_db_path()}')

In [ ]:
# The 10 hexapod LUT axes, in database order. Ordering and units are documented at
# aos/code/aos_trim.py:fetch_hexapod_lut_for_visits -- M2 first (salIndex 2), then camera
# (salIndex 1), each z/x/y/u/v. NOTE the angular axes are deg here, not the arcsec the
# Trim uses.
LUT_LABELS = ['M2_LUT_dz', 'M2_LUT_dx', 'M2_LUT_dy', 'M2_LUT_u', 'M2_LUT_v',
              'Cam_LUT_dz', 'Cam_LUT_dx', 'Cam_LUT_dy', 'Cam_LUT_u', 'Cam_LUT_v']
LUT_UNITS = ['um', 'um', 'um', 'deg', 'deg'] * 2

TRIM_COLS = [f'dof{i}' for i in range(50)]
LUT_COLS = [f'lut_dof{i}' for i in range(10)]

# Trim grid pages: (title, DOF indices, panel grid). The hexapod page keeps the two
# hexapods on their own rows; the bending pages are 4x5.
TRIM_PAGES = [
    ('Trim, hexapod axes (DOF 0-9): M2 then camera, z/x/y/rx/ry',
     DOF_GROUPS['m2_hex'] + DOF_GROUPS['cam_hex'], (2, 5)),
    ('Trim, M1M3 bending modes (DOF 10-29)', DOF_GROUPS['m1m3_bending'], (4, 5)),
    ('Trim, M2 bending modes (DOF 30-49)', DOF_GROUPS['m2_bending'], (4, 5)),
]

print('DOF 0-9 :', ', '.join(f'{i}={LABELS_50DOF[i]} [{DOF_UNITS_50[i]}]' for i in range(10)))

<a id='functions'></a>
## 3. Helper functions

In [ ]:
def night_colors(nights, cmap='viridis'):
    """One colour per night, in date order.

    Parameters
    ----------
    nights : `array_like` [`int`]
        ``day_obs`` values as ``YYYYMMDD``.
    cmap : `str`, optional

    Returns
    -------
    colors : `dict`
        Maps each ``day_obs`` to an RGBA tuple.
    """
    nights = sorted(int(n) for n in set(nights))
    cm = plt.get_cmap(cmap)
    if len(nights) == 1:
        return {nights[0]: cm(0.5)}
    return {n: cm(i / (len(nights) - 1)) for i, n in enumerate(nights)}


def grid_vs_seqnum(df, cols, labels, units, title, grid, colors,
                   figsize=GRID_FIGSIZE, ms=MS):
    """Grid of panels, one commanded quantity each, against ``seq_num`` per night.

    Parameters
    ----------
    df : `pandas.DataFrame`
        Selected visits, carrying ``day_obs``, ``seq_num`` and every column in `cols`.
    cols : `list` [`str`]
        Columns to plot, one per panel.
    labels : `list` [`str`]
        Panel titles, parallel to `cols`.
    units : `list` [`str`]
        Unit of each column -- µm for a translation, arcsec or deg for a rotation --
        parallel to `cols`. A panel with no unit is a bug, so this is not optional.
    title : `str`
        Figure suptitle.
    grid : `tuple` [`int`]
        ``(nrow, ncol)``.
    colors : `dict`
        From `night_colors`.
    figsize : `tuple` [`float`], optional
    ms : `float`, optional

    Returns
    -------
    fig : `matplotlib.figure.Figure`

    Notes
    -----
    Each night is drawn as its own series against that night's ``seq_num``, so a night with
    a different sequence numbering does not connect across the daytime gap. A panel whose
    column is entirely NaN is annotated as such rather than left as empty axes, because a
    missing telemetry channel and a flat commanded value look identical otherwise.
    """
    nrow, ncol = grid
    fig, axes = plt.subplots(nrow, ncol, figsize=figsize, sharex=True, squeeze=False)
    flat = axes.ravel()
    for ax, col, lab, unit in zip(flat, cols, labels, units):
        v = df[col].to_numpy(float)
        if not np.isfinite(v).any():
            ax.text(0.5, 0.5, f'{lab}\nno finite values', ha='center', va='center',
                    transform=ax.transAxes, fontsize=8, color='0.4')
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        for night, g in df.groupby('day_obs'):
            ax.plot(g.seq_num, g[col], 'o-', ms=ms, lw=0.8,
                    color=colors[int(night)], label=str(int(night)))
        finite = v[np.isfinite(v)]
        ax.set_title(f'{lab} [{unit}]\nspan {np.ptp(finite):.4g}, '
                     f'nMAD {nmad(finite):.4g} {unit}', fontsize=8)
        ax.tick_params(labelsize=7)
    for ax in flat[len(cols):]:
        ax.axis('off')
    for ax in axes[-1]:
        ax.set_xlabel('seq_num', fontsize=8)
    # One legend for the figure; the per-panel labels repeat every night.
    h, l = flat[0].get_legend_handles_labels()
    if h:
        fig.legend(h, l, loc='lower center', ncol=min(len(l), 8), fontsize=8,
                   frameon=False, title='day_obs', title_fontsize=8)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    return fig


def dof_span_table(df, cols, labels, units, min_span=0.0):
    """Per-column range and robust scatter over the selected visits.

    Parameters
    ----------
    df : `pandas.DataFrame`
    cols, labels, units : `list` [`str`]
        Parallel lists: column name, display name, and the unit the values carry.
    min_span : `float`, optional
        Keep only rows whose span exceeds this, in each row's own unit. Use it to find
        which DOF actually moved.

    Returns
    -------
    out : `pandas.DataFrame`
        ``dof``, ``label``, ``unit``, ``n_finite``, ``median``, ``span``, ``nmad``, each
        value in the unit named in its own ``unit`` column.
    """
    rows = []
    for col, lab, unit in zip(cols, labels, units):
        v = df[col].to_numpy(float)
        v = v[np.isfinite(v)]
        rows.append(dict(dof=col, label=lab, unit=unit, n_finite=len(v),
                         median=np.median(v) if len(v) else np.nan,
                         span=np.ptp(v) if len(v) else np.nan,
                         nmad=nmad(v) if len(v) else np.nan))
    out = pd.DataFrame(rows)
    return out[out.span > min_span] if min_span else out

<a id='data'></a>
## 4. Load the telemetry and select the programs

`visit_telemetry` carries no `science_program` column, so the BLOCK selection comes from
the Consolidated Database (ConsDB) `exposure` table via `efd_db.join_consdb`. That is one
query per night over the whole database, so the result is cached to parquet and reused.

In [ ]:
cols = ['visit_id', 'day_obs', 'seq_num'] + LUT_COLS + TRIM_COLS
tel = efd_db.visits(day_obs_range=DAY_OBS_RANGE, columns=cols)
print(f'visit_telemetry rows: {len(tel)}, '
      f'day_obs {int(tel.day_obs.min())} to {int(tel.day_obs.max())}, '
      f'{tel.day_obs.nunique()} nights')

In [ ]:
cache = OUT_DIR / CACHE_PATH
if CACHE_META and cache.exists():
    meta = pd.read_parquet(cache)
    print(f'read cached ConsDB metadata for {len(meta)} visits from {cache}')
else:
    # groups=('meta',) is the exposure table alone: band, img_type, science_program,
    # pointing. The temperature/wind/image-quality groups are not needed here.
    joined = efd_db.join_consdb(tel[['visit_id', 'day_obs', 'seq_num']], groups=('meta',))
    # join_consdb renames the pointing columns (altitude -> altitude_deg,
    # azimuth -> azimuth_deg_consdb), so keep whichever of the wanted set is present
    # rather than naming them positionally.
    want = ['visit_id', 'day_obs', 'seq_num', 'band', 'img_type', 'science_program',
            'altitude_deg', 'azimuth_deg_consdb']
    meta = joined[[c for c in want if c in joined.columns]].copy()
    if CACHE_META:
        cache.parent.mkdir(parents=True, exist_ok=True)
        meta.to_parquet(cache)
        print(f'wrote {cache}')
print(f'metadata rows: {len(meta)}')

In [ ]:
prog = meta.science_program.astype(str)
sel = prog.str.contains('|'.join(PROGRAMS), case=False, na=False)
if IMG_TYPES is not None:
    sel &= meta.img_type.astype(str).str.lower().isin([t.lower() for t in IMG_TYPES])

_keep = [c for c in ['visit_id', 'band', 'img_type', 'science_program',
                     'altitude_deg', 'azimuth_deg_consdb'] if c in meta.columns]
df = (tel.merge(meta[sel][_keep], on='visit_id', how='inner')
         .sort_values(['day_obs', 'seq_num'])
         .reset_index(drop=True))

found = sorted(df.science_program.unique())
missing = [p for p in PROGRAMS if not any(p.lower() in f.lower() for f in found)]
print(f'selected {len(df)} visits over {df.day_obs.nunique()} nights')
print(f'programs found  : {", ".join(found) if found else "none"}')
if missing:
    print(f'programs with NO visits in this database: {", ".join(missing)}')
COLORS = night_colors(df.day_obs)

<a id='inventory'></a>
## 5. Sample inventory

What turned up, and how complete the two commanded groups are. A visit present in
`visit_telemetry` can still have a NaN LUT or Trim if the underlying Engineering Facility
Database (EFD) event was missing for that night, so the coverage line matters before
reading any plot.

In [ ]:
if len(df):
    inv = (df.groupby(['science_program', 'day_obs', 'band'])
             .agg(n_visits=('visit_id', 'size'),
                  seq_first=('seq_num', 'min'), seq_last=('seq_num', 'max'),
                  elev_min_deg=('altitude_deg', 'min'),
                  elev_max_deg=('altitude_deg', 'max'))
             .reset_index())
    display(inv)

    n_lut = int(df[LUT_COLS].notna().all(axis=1).sum())
    n_trim = int(df[TRIM_COLS].notna().all(axis=1).sum())
    print(f'visits with all 10 LUT axes finite : {n_lut} of {len(df)} '
          f'({100 * n_lut / len(df):.1f}%)')
    print(f'visits with all 50 Trim DOF finite : {n_trim} of {len(df)} '
          f'({100 * n_trim / len(df):.1f}%)')
else:
    print('no visits selected -- check PROGRAMS, IMG_TYPES and DAY_OBS_RANGE')

<a id='lut'></a>
## 6. Hexapod LUT, all 10 axes

One panel per axis against `seq_num`, coloured by night. `Cam_LUT_dz` (index 5) is the
filter-dependent focus term and is usually the only axis with real structure.

In [ ]:
fig_lut = grid_vs_seqnum(
    df, LUT_COLS, LUT_LABELS, LUT_UNITS,
    f'Hexapod LUT (MTHexapod compensationOffset) against seq_num — '
    f'{", ".join(PROGRAMS)}, {len(df)} visits.  Angular axes are deg, not arcsec',
    (2, 5), COLORS)
plt.show()

In [ ]:
print('LUT axes that moved, sorted by span (each row in its own unit):')
display(dof_span_table(df, LUT_COLS, LUT_LABELS, LUT_UNITS)
        .sort_values('span', ascending=False))

<a id='trim'></a>
## 7. Trim, all 50 DOF

Three pages: the ten hexapod axes, the twenty M1M3 bending modes, and the twenty M2
bending modes. Units are µm for the translations and bending amplitudes, arcsec for the
hexapod rotations — each panel title carries its own.

In [ ]:
trim_figs = []
for title, idx, grid in TRIM_PAGES:
    fig = grid_vs_seqnum(df, [f'dof{i}' for i in idx],
                         [f'{i}: {LABELS_50DOF[i]}' for i in idx],
                         [DOF_UNITS_50[i] for i in idx],
                         f'{title} — {", ".join(PROGRAMS)}, {len(df)} visits',
                         grid, COLORS)
    trim_figs.append(fig)
    plt.show()

In [ ]:
# Which DOF actually moved. The commanded bending modes of the test should stand out here.
spans = dof_span_table(df, TRIM_COLS, LABELS_50DOF, list(DOF_UNITS_50))
print('Trim DOF with the largest span over the selected visits, each row in '
      'its own unit:')
display(spans.sort_values('span', ascending=False).head(15))

<a id='single'></a>
## 8. A single DOF in detail

`SINGLE_DOF` on its own, with the matching LUT axis beside it where one exists (that is,
for DOF 0–9 only). This is the view for asking what the LUT had loaded while a given DOF
was being commanded.

In [ ]:
i = SINGLE_DOF
has_lut = i < 10
fig_single, axes = plt.subplots(2 if has_lut else 1, 1, figsize=(13.0, 6.5 if has_lut else 3.6),
                               sharex=True, squeeze=False)
ax = axes[0][0]
for night, g in df.groupby('day_obs'):
    ax.plot(g.seq_num, g[f'dof{i}'], 'o-', ms=5, lw=0.9,
            color=COLORS[int(night)], label=str(int(night)))
ax.set_ylabel(f'Trim {LABELS_50DOF[i]} [{DOF_UNITS_50[i]}]')
ax.set_title(f'Trim DOF {i} = {LABELS_50DOF[i]}, and the matching hexapod LUT axis'
             if has_lut else f'Trim DOF {i} = {LABELS_50DOF[i]}')
ax.legend(fontsize=8, ncol=6, title='day_obs', title_fontsize=8)

if has_lut:
    ax2 = axes[1][0]
    for night, g in df.groupby('day_obs'):
        ax2.plot(g.seq_num, g[f'lut_dof{i}'], 's-', ms=5, lw=0.9,
                 color=COLORS[int(night)])
    ax2.set_ylabel(f'{LUT_LABELS[i]} [{LUT_UNITS[i]}]')
    ax2.set_xlabel('seq_num')
else:
    ax.set_xlabel('seq_num')
    print(f'DOF {i} = {LABELS_50DOF[i]} is a bending mode; the hexapod LUT has no '
          f'matching axis (it covers DOF 0-9 only).')
fig_single.tight_layout()
plt.show()

<a id='pdf'></a>
## 9. Optional PDF

In [ ]:
if SAVE_PDF:
    pdf_path = OUT_DIR / 'bounce' / 'bending_mode_test_lut_trim.pdf'
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(pdf_path) as pdf:
        for f in [fig_lut, *trim_figs, fig_single]:
            pdf.savefig(f)
    print(f'wrote {pdf_path}')
else:
    print('SAVE_PDF is False; nothing written')